In [0]:
# DLT work with three types
# 1. streaming tables 
# 2. views
# 3. materialized views
from pyspark import pipelines as dp
from pyspark.sql.functions import count, current_timestamp

In [0]:

# =====================================================
# 3. JOIN ORDERS AND CUSTOMERS - VIEW
# =====================================================

@dp.temporary_view(
    name="orders_customers_v"
)
def orders_customers_v():

    orders = spark.read.table("LIVE.orders_st").alias("o")

    customers = spark.read.table("LIVE.customers_mv").alias("c")

    return (
        orders
        .join(
            customers,
            orders.o_custkey == customers.c_custkey,
            "left"
        )
        .select(
            # Order information
            orders.o_orderkey,
            orders.o_orderstatus,
            orders.o_orderdate,
            orders.o_totalprice,
            orders.o_orderpriority,

            # Customer information
            customers.c_custkey,
            customers.c_name,
            customers.c_address,
            customers.c_phone,
            customers.c_acctbal,
            customers.c_mktsegment
        )
    )


In [0]:
# =====================================================
# 4. STREAMING TABLE - GOLD
# =====================================================
@dp.table(
    name="orders_gold_st",
    comment="Streaming ingestion of gold layer"
)
def func_orders_gold_st():
    df = spark.read.table("LIVE.orders_customers_v")

    df_final = (
        df.groupBy("c_mktsegment")
        .agg(count("o_orderkey").alias("total_orders"))
        .withColumn("__insert_date", current_timestamp())
    )

    return df_final